# Exercise 03 — Prompt Engineering & Evaluation

Topics covered:
1. **Prompt evaluation pipeline** — dataset generation, running evals, grading
2. **Code-based grading** — validate JSON/Python syntax automatically
3. **Model-based grading** — ask Claude to score its own outputs
4. **Prompt engineering techniques** — clear & direct, specific, XML tags, few-shot examples

**Goal:** Build an AWS coding assistant prompt that outputs only raw Python, JSON, or regex — no explanations.

In [ ]:
import os
import json
import ast
import re
from dotenv import load_dotenv
import anthropic

load_dotenv(dotenv_path="../.env")
client = anthropic.Anthropic()

# Use Haiku for dataset generation and grading (fast + cheap)
FAST_MODEL = "claude-haiku-4-5"
# Use Sonnet for the assistant being evaluated
MAIN_MODEL = "claude-sonnet-4-5"

print("Ready!")

## Part 1 — Generating a Test Dataset

Instead of writing test cases by hand, we can ask Claude (Haiku) to generate them for us.

Each test case is a JSON object with a `task` field (what the user is asking) and a `format` field (expected output format: `Python`, `JSON`, or `RegEx`).

In [ ]:
DATASET_GENERATION_PROMPT = """Generate a dataset of 9 AWS coding assistance tasks.
Return a JSON array where each object has:
- "task": a specific AWS coding task (e.g. "Write a Python function to list all S3 buckets")
- "format": one of "Python", "JSON", or "RegEx" (what format the output should be in)

Include exactly 3 tasks for each format.
Tasks should be realistic AWS use cases."""


def generate_dataset() -> list[dict]:
    """Ask Claude to generate a list of test cases."""
    response = client.messages.create(
        model=FAST_MODEL,
        max_tokens=1000,
        stop_sequences=["```"],
        messages=[
            {"role": "user",      "content": DATASET_GENERATION_PROMPT},
            {"role": "assistant", "content": "```json\n"}
        ]
    )
    return json.loads(response.content[0].text.strip())


dataset = generate_dataset()
print(f"Generated {len(dataset)} test cases:\n")
for i, case in enumerate(dataset, 1):
    print(f"{i}. [{case['format']:6s}] {case['task']}")

## Part 2 — Running the Eval

Three core functions:

1. `run_prompt(test_case, prompt_template)` — merge test case into prompt, call Claude, return output
2. `run_test_case(test_case, prompt_template)` — run prompt + grade result, return summary dict
3. `run_eval(dataset, prompt_template)` — loop through all cases, collect results

In [ ]:
# ── v1 Prompt: VERY basic (this will score poorly on purpose) ──────────────
PROMPT_V1 = "Please solve the following task: {task}"


def run_prompt(test_case: dict, prompt_template: str) -> str:
    """Merge test case into prompt and call the model."""
    prompt = prompt_template.format(**test_case)
    response = client.messages.create(
        model=MAIN_MODEL,
        max_tokens=500,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text


# Quick sanity check
sample_output = run_prompt(dataset[0], PROMPT_V1)
print(f"Task: {dataset[0]['task']}\n")
print(f"Output (v1 prompt):\n{sample_output[:500]}")

## Part 3 — Code-Based Grading

For code output we can automatically check syntax validity:
- `validate_json()` — tries `json.loads()`, score 10 if valid, 0 if not
- `validate_python()` — tries `ast.parse()`, score 10 if valid, 0 if not
- `validate_regex()` — tries `re.compile()`, score 10 if valid, 0 if not

In [ ]:
def validate_json(text: str) -> int:
    try:
        json.loads(text)
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text: str) -> int:
    try:
        ast.parse(text)
        return 10
    except SyntaxError:
        return 0


def validate_regex(text: str) -> int:
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def syntax_score(text: str, expected_format: str) -> int:
    """Route to the correct validator based on expected format."""
    validators = {
        "Python": validate_python,
        "JSON":   validate_json,
        "RegEx":  validate_regex,
    }
    validator = validators.get(expected_format)
    if not validator:
        raise ValueError(f"Unknown format: {expected_format}")
    return validator(text)


# Test the validators
print("Valid JSON:   ", validate_json('{"key": "value"}'))
print("Invalid JSON: ", validate_json('{key: value}'))
print("Valid Python: ", validate_python('def hello():\n    return "world"'))
print("Invalid  py:  ", validate_python('def hello( :'))

## Part 4 — Model-Based Grading

For semantic quality we ask Claude (Haiku) to score the output. We ask for strengths, weaknesses, and a numerical score to avoid "middle score" bias.

In [ ]:
GRADER_PROMPT = """You are evaluating an AI assistant's response to an AWS coding task.

Task: {task}
Expected format: {format}
Response to grade:
---
{output}
---

Grade the response on these criteria:
1. Does it contain ONLY the requested {format} code with NO explanations or commentary?
2. Is the code correct and would it work for the task?
3. Is the code clean and idiomatic?

Respond with JSON containing: strengths, weaknesses, score (1-10)."""


def model_grade(test_case: dict, output: str) -> dict:
    """Ask Claude Haiku to score a model output. Returns {score, strengths, weaknesses}."""
    grader_prompt = GRADER_PROMPT.format(
        task=test_case["task"],
        format=test_case["format"],
        output=output
    )
    response = client.messages.create(
        model=FAST_MODEL,
        max_tokens=512,
        stop_sequences=["```"],
        messages=[
            {"role": "user",      "content": grader_prompt},
            {"role": "assistant", "content": "```json\n"}
        ]
    )
    return json.loads(response.content[0].text.strip())


# Test the grader
test_output = sample_output
grade = model_grade(dataset[0], test_output)
print(f"Model grade: {grade['score']}/10")
print(f"Strengths: {grade.get('strengths', 'N/A')}")
print(f"Weaknesses: {grade.get('weaknesses', 'N/A')}")

In [ ]:
def run_test_case(test_case: dict, prompt_template: str) -> dict:
    """Run a single test case and return a result summary."""
    output = run_prompt(test_case, prompt_template)
    m_grade = model_grade(test_case, output)
    s_score = syntax_score(output.strip(), test_case["format"])
    final_score = (m_grade["score"] + s_score) / 2
    return {
        "task":         test_case["task"],
        "format":       test_case["format"],
        "output":       output,
        "model_score":  m_grade["score"],
        "syntax_score": s_score,
        "final_score":  final_score,
        "weaknesses":   m_grade.get("weaknesses", ""),
    }


def run_eval(dataset: list[dict], prompt_template: str) -> list[dict]:
    """Run all test cases and print a summary."""
    results = []
    for i, case in enumerate(dataset, 1):
        print(f"Running case {i}/{len(dataset)}...", end=" ")
        result = run_test_case(case, prompt_template)
        results.append(result)
        print(f"score={result['final_score']:.1f}")

    avg = sum(r["final_score"] for r in results) / len(results)
    print(f"\n{'='*40}")
    print(f"Average score: {avg:.2f}/10")
    return results


print("Functions defined. Ready to run evaluations.")

## Part 5 — Prompt Engineering: Improving the Score

Let's apply prompt engineering techniques one at a time and measure improvement.

### Baseline: v1 prompt (just the task)

Expect a low score — Claude will include explanations and markdown.

In [ ]:
# Baseline evaluation — v1 prompt
print("=== Evaluating v1 prompt (baseline) ===")
results_v1 = run_eval(dataset, PROMPT_V1)

### Technique 1 — Clear & Direct

- Start with an **action verb**
- Specify exactly **what** you want and **what format**
- No filler words

In [ ]:
PROMPT_V2 = """Write {format} code to complete the following AWS task.
Output ONLY the raw {format} code with no explanations, comments, or markdown.

Task: {task}"""

print("=== Evaluating v2 prompt (clear & direct) ===")
results_v2 = run_eval(dataset, PROMPT_V2)

### Technique 2 — Be Specific (Add Guidelines)

Add a bulleted list of output requirements (Type A attributes) and processing steps (Type B steps).

In [ ]:
PROMPT_V3 = """Write {format} code to complete the following AWS task.
Output ONLY the raw {format} code. No explanations, no comments, no markdown fences.

Requirements:
- The output must be valid, executable {format} code
- Include only what is necessary to solve the task — no boilerplate
- Follow AWS best practices and idiomatic {format} style
- For Python: use boto3 where appropriate
- For JSON: use valid JSON syntax with proper types
- For RegEx: provide only the pattern string, no surrounding code

Task: {task}"""

print("=== Evaluating v3 prompt (specific guidelines) ===")
results_v3 = run_eval(dataset, PROMPT_V3)

### Technique 3 — XML Tags + Few-Shot Example

Wrap the task in an XML tag so Claude clearly distinguishes it from the instructions. Add an example showing expected input/output.

In [ ]:
PROMPT_V4 = """Write {format} code to complete the AWS task inside <task> tags.
Output ONLY the raw {format} code — no explanations, no comments, no markdown fences.

Requirements:
- The output must be valid, executable {format} code
- Include only what is necessary to solve the task — no boilerplate
- Follow AWS best practices and idiomatic {format} style
- For Python: use boto3 where appropriate
- For JSON: use valid JSON syntax with proper types
- For RegEx: provide only the pattern string, no surrounding code

<example>
<task>Write Python code to list all S3 buckets</task>
<ideal_output>
import boto3
s3 = boto3.client('s3')
response = s3.list_buckets()
buckets = [b['Name'] for b in response.get('Buckets', [])]
</ideal_output>
Notice: no explanation, no markdown, just the code.
</example>

<task>{task}</task>"""

print("=== Evaluating v4 prompt (XML tags + few-shot) ===")
results_v4 = run_eval(dataset, PROMPT_V4)

## Part 6 — Compare All Versions

In [ ]:
def avg_score(results: list[dict]) -> float:
    return sum(r["final_score"] for r in results) / len(results)


print("\n" + "="*45)
print("PROMPT VERSION COMPARISON")
print("="*45)
versions = [
    ("v1 — basic",              results_v1),
    ("v2 — clear & direct",     results_v2),
    ("v3 — specific",           results_v3),
    ("v4 — XML tags + few-shot",results_v4),
]
for name, results in versions:
    score = avg_score(results)
    bar = "█" * int(score)
    print(f"{name:<28} {score:.2f}/10  {bar}")

### Exercise — Write Your Own Improved Prompt

Based on the weaknesses you see in the results, write a `PROMPT_V5` that scores higher than v4. You can:
- Add more examples
- Add stricter instructions
- Change the framing

In [ ]:
PROMPT_V5 = """TODO: Write your improved prompt here.

Task: {task}
Format: {format}"""

print("=== Evaluating v5 prompt (your version) ===")
results_v5 = run_eval(dataset, PROMPT_V5)

print(f"\nYour improvement: {avg_score(results_v4):.2f} → {avg_score(results_v5):.2f}")

## Summary

- ✅ Generated a test dataset using Claude itself
- ✅ Built a code-based grader (JSON/Python/RegEx syntax validation)
- ✅ Built a model-based grader (semantic scoring via Claude)
- ✅ Applied 4 prompt engineering techniques and measured score improvement

**Key insight:** Prompt engineering without evaluation is guesswork. Using an eval pipeline gives you objective scores to compare versions systematically.

**Next:** [04_tool_use_reminder_bot.ipynb](04_tool_use_reminder_bot.ipynb) — giving Claude tools to call external functions